# Notebook 78: V8.9 Ring-2 Operator Freeze, Parent Equivalence and 151-Page HITL Regeneration

Fail-closed release notebook for `v8_9_operator.py`. It proves exact V8.8 parent equivalence, reproduces Notebook 75 C1, runs exact replay, and regenerates the complete 151-page HITL cohort. No clean/truth data is used by the operator.


In [7]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy scikit-learn joblib torch torchvision pyarrow


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, importlib.util, sys
import cv2, numpy as np, pandas as pd
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
OPERATOR_SOURCE=ROOT/'v8_9_release/v8_9_operator.py'
P_PRINT_BUNDLE=ROOT/'observed_print_full_development_freeze_v8_2f'/'20261117_120424_v8_2f_full_development_p_print_freeze'/'p_print_provider'
REMOVE_BUNDLE=ROOT/'m5_remove_full_development_freeze_v8_8f'/'20261117_110512_notebook52f_v8_8_full_development_remove_freeze'/'full_development_remove'
NB72_ROOT=ROOT/'notebook72_frozen_v8_8_annotated_only_operational_trial'
NB75_ROOT=ROOT/'notebook75_v88_collar_preservation_counterfactual'
HITL_EXPECTED_PAGES=151
RUN=ROOT/'v8_9_release'/ (datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook78')
for p in [RUN,RUN/'gates',RUN/'pages',RUN/'audit',RUN/'replay',RUN/'upload']: p.mkdir(parents=True,exist_ok=False)
def require(x,m):
 if not bool(x): raise RuntimeError(m)
def sha256_file(p):
 h=hashlib.sha256();
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''): h.update(b)
 return h.hexdigest()
def latest(root,required):
 hits=[p for p in Path(root).rglob('*') if p.is_dir() and all((p/r).exists() for r in required)]
 return max(hits,key=lambda p:p.stat().st_mtime) if hits else None
require(OPERATOR_SOURCE.exists(),f'Missing {OPERATOR_SOURCE}')
spec=importlib.util.spec_from_file_location('v8_9_operator',OPERATOR_SOURCE);
mod=importlib.util.module_from_spec(spec);
sys.modules['v8_9_operator']=mod
spec.loader.exec_module(mod)
op=mod.load_v89_operator(P_PRINT_BUNDLE,REMOVE_BUNDLE)
print('Run:',RUN)


Run: /content/drive/MyDrive/MusicAnnotationProject/v8_9_release/20261119_173913_notebook78


## Gate 1: exact V8.8 parent equivalence

In [10]:
# ============================================================
# Gate 1: exact V8.8 parent equivalence
# ============================================================

NB72 = latest(
    NB72_ROOT,
    ["inputs/runtime_roster.csv"],
)

require(
    NB72 is not None,
    "Complete Notebook 72 run not found.",
)

roster = pd.read_csv(
    NB72 / "inputs/runtime_roster.csv"
)

required = {
    "page",
    "observed_path",
    "seed_mask_path",
}

require(
    required <= set(roster.columns),
    (
        "Notebook 72 runtime roster missing: "
        f"{sorted(required - set(roster.columns))}"
    ),
)

rows = []

for _, r in roster.iterrows():

    page = str(r.page)

    obs = cv2.imread(
        str(r.observed_path),
        cv2.IMREAD_GRAYSCALE,
    )

    seed = cv2.imread(
        str(r.seed_mask_path),
        cv2.IMREAD_GRAYSCALE,
    )

    require(
        obs is not None
        and seed is not None
        and obs.shape == seed.shape,
        f"{page}: input failure",
    )

    # --------------------------------------------------------
    # Run the V8.9 operator once.
    #
    # V89PageResult already contains the exact parent V8.8
    # corrected page as parent_v88_corrected.
    # --------------------------------------------------------

    child = op.run_page(
        obs,
        seed,
        page=page,
    )

    ring2 = np.asarray(
        child.ring2_mask,
        dtype=bool,
    )

    delta = np.asarray(
        child.v89_delta_mask,
        dtype=bool,
    )


    # --------------------------------------------------------
    # Core V8.9 parent-regression contract
    # --------------------------------------------------------

    non_ring2_mismatch = int(
        np.sum(
            child.corrected[~ring2]
            != child.parent_v88_corrected[~ring2]
        )
    )

    ring2_observed_mismatch = int(
        np.sum(
            child.corrected[ring2]
            != child.observed[ring2]
        )
    )

    unexpected_delta = int(
        np.sum(
            delta
            & ~ring2
        )
    )

    expected_effective_delta = (
        ring2
        & (
            child.observed
            != child.parent_v88_corrected
        )
    )

    delta_identity_mismatch = int(
        np.logical_xor(
            delta,
            expected_effective_delta,
        ).sum()
    )


    # --------------------------------------------------------
    # Internal V8.8 result sanity
    #
    # These arrays are produced by the parent implementation
    # inside the V8.9 call. They do not need a second expensive
    # inference call merely to prove that V8.9 retained them.
    # --------------------------------------------------------

    finite_p_print = np.isfinite(
        child.p_print
    )

    finite_p_remove = np.isfinite(
        child.p_remove
    )

    require(
        child.p_print.shape == obs.shape,
        f"{page}: p_print shape mismatch",
    )

    require(
        child.p_remove.shape == obs.shape,
        f"{page}: p_remove shape mismatch",
    )

    require(
        child.m3_mask.shape == obs.shape,
        f"{page}: M3 shape mismatch",
    )

    require(
        child.safe_remove_mask.shape == obs.shape,
        f"{page}: SAFE_REMOVE shape mismatch",
    )

    require(
        child.m5_remove_mask.shape == obs.shape,
        f"{page}: M5 shape mismatch",
    )


    # --------------------------------------------------------
    # Fail closed
    # --------------------------------------------------------

    require(
        non_ring2_mismatch == 0,
        (
            f"{page}: V8.9 changed "
            f"{non_ring2_mismatch} non-Ring-2 "
            "V8.8 parent pixels."
        ),
    )

    require(
        ring2_observed_mismatch == 0,
        (
            f"{page}: V8.9 failed observed-byte "
            f"preservation at "
            f"{ring2_observed_mismatch} Ring-2 pixels."
        ),
    )

    require(
        unexpected_delta == 0,
        (
            f"{page}: V8.9 changed "
            f"{unexpected_delta} pixels outside Ring 2."
        ),
    )

    require(
        delta_identity_mismatch == 0,
        (
            f"{page}: V8.9 effective-delta map "
            f"mismatch at {delta_identity_mismatch} pixels."
        ),
    )


    rows.append({
        "page":
            page,

        "ring2_pixels":
            int(ring2.sum()),

        "effective_delta_pixels":
            int(delta.sum()),

        "parent_non_ring2_mismatch":
            non_ring2_mismatch,

        "ring2_observed_mismatch":
            ring2_observed_mismatch,

        "unexpected_delta_outside_ring2":
            unexpected_delta,

        "delta_identity_mismatch":
            delta_identity_mismatch,

        "p_print_finite_pixels":
            int(finite_p_print.sum()),

        "p_print_nan_pixels":
            int(np.isnan(child.p_print).sum()),

        "p_remove_finite_pixels":
            int(finite_p_remove.sum()),

        "p_remove_nan_pixels":
            int(np.isnan(child.p_remove).sum()),
    })

    print(
        f"{page}: "
        f"Ring2={int(ring2.sum()):,}, "
        f"changed={int(delta.sum()):,}"
    )


eq = pd.DataFrame(rows)

eq.to_csv(
    RUN / "gates" / "v88_parent_equivalence.csv",
    index=False,
)


# ------------------------------------------------------------
# Final Gate-1 assertion
# ------------------------------------------------------------

equivalence_columns = [
    "parent_non_ring2_mismatch",
    "ring2_observed_mismatch",
    "unexpected_delta_outside_ring2",
    "delta_identity_mismatch",
]

require(
    (
        eq[equivalence_columns]
        .to_numpy()
        == 0
    ).all(),
    (
        "V8.8 parent-equivalence gate failed. "
        "Inspect v88_parent_equivalence.csv."
    ),
)

print()
print("=" * 72)
print("PASS: V8.8 -> V8.9 PARENT EQUIVALENCE")
print("=" * 72)

print(
    "Pages:",
    len(eq),
)

print(
    "Ring-2 pixels:",
    f"{int(eq.ring2_pixels.sum()):,}",
)

print(
    "Effective V8.9 changed pixels:",
    f"{int(eq.effective_delta_pixels.sum()):,}",
)

print(
    "Non-Ring-2 V8.8 regressions:",
    int(eq.parent_non_ring2_mismatch.sum()),
)

print(
    "Ring-2 observed-byte mismatches:",
    int(eq.ring2_observed_mismatch.sum()),
)

print(
    "Unexpected changes outside Ring 2:",
    int(eq.unexpected_delta_outside_ring2.sum()),
)

print(
    "Delta-map identity mismatches:",
    int(eq.delta_identity_mismatch.sum()),
)

Cartoon Classics - parts_Page_01: Ring2=5,078, changed=4,864
Cartoon Classics - parts_Page_02: Ring2=4,701, changed=4,448
Cartoon Classics - parts_Page_03: Ring2=1,377, changed=1,358
Cartoon Classics - parts_Page_04: Ring2=4,884, changed=4,526
Cartoon Classics - parts_Page_05: Ring2=11,856, changed=11,340
Cartoon Classics - parts_Page_06: Ring2=4,780, changed=4,641
Cartoon Classics - parts_Page_07: Ring2=11,169, changed=10,487
Cartoon Classics - parts_Page_08: Ring2=16,645, changed=14,746
Cartoon Classics - parts_Page_09: Ring2=7,808, changed=7,602
Cartoon Classics - parts_Page_10: Ring2=11,692, changed=10,810
Cartoon Classics - parts_Page_11: Ring2=24,771, changed=21,582
Cartoon Classics - parts_Page_12: Ring2=7,838, changed=7,596
Cartoon Classics - parts_Page_13: Ring2=7,444, changed=6,810
Cartoon Classics - parts_Page_14: Ring2=12,544, changed=11,571
Cartoon Classics - parts_Page_15: Ring2=8,537, changed=7,641
Cartoon Classics - parts_Page_16: Ring2=10,293, changed=9,338
Cartoon Cla

## Gate 2: Notebook 75 C1 exact reproduction

In [11]:
# ============================================================
# Gate 2: Notebook 75 C1 SEMANTIC reproduction
#
# IMPORTANT:
# Notebook 75's rendered C1 pages used the historical Notebook72
# incumbent output, which was subsequently superseded by the
# corrected V8.8 assembly.
#
# Therefore V8.9 is NOT required to reproduce Notebook75's
# complete C1 page bytes.
#
# What must reproduce exactly is the C1 transformation:
#
#     Ring2      -> exact observed byte
#     not Ring2  -> exact current incumbent byte
#
# Here the current incumbent is the corrected V8.8 parent.
# ============================================================

rows = []

for _, r in roster.iterrows():

    page = str(r.page)

    obs = cv2.imread(
        str(r.observed_path),
        cv2.IMREAD_GRAYSCALE,
    )

    seed = cv2.imread(
        str(r.seed_mask_path),
        cv2.IMREAD_GRAYSCALE,
    )

    require(
        obs is not None
        and seed is not None
        and obs.shape == seed.shape,
        f"{page}: input failure",
    )

    result = op.run_page(
        obs,
        seed,
        page=page,
    )

    ring2 = np.asarray(
        result.ring2_mask,
        dtype=bool,
    )

    parent = np.asarray(
        result.parent_v88_corrected,
    )

    child = np.asarray(
        result.corrected,
    )


    # --------------------------------------------------------
    # Independently rebuild Notebook75 C1 geometry.
    #
    # Ring1 = dilation1(seed) - seed
    # Ring2 = dilation2(seed) - dilation1(seed)
    # --------------------------------------------------------

    seed_bool = seed > 0

    k1 = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (3, 3),
    )

    k2 = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (5, 5),
    )

    d1 = (
        cv2.dilate(
            seed_bool.astype(np.uint8),
            k1,
        ) > 0
    )

    d2 = (
        cv2.dilate(
            seed_bool.astype(np.uint8),
            k2,
        ) > 0
    )

    expected_ring2 = d2 & ~d1


    # --------------------------------------------------------
    # Exact Notebook75 C1 transformation, but applied to the
    # corrected V8.8 parent.
    # --------------------------------------------------------

    expected_c1 = parent.copy()

    expected_c1[
        expected_ring2
    ] = obs[
        expected_ring2
    ]


    # --------------------------------------------------------
    # Equivalence checks
    # --------------------------------------------------------

    ring2_xor = int(
        np.logical_xor(
            ring2,
            expected_ring2,
        ).sum()
    )

    output_diff = int(
        np.sum(
            child != expected_c1
        )
    )

    ring2_observed_mismatch = int(
        np.sum(
            child[expected_ring2]
            != obs[expected_ring2]
        )
    )

    non_ring2_parent_mismatch = int(
        np.sum(
            child[~expected_ring2]
            != parent[~expected_ring2]
        )
    )


    require(
        ring2_xor == 0,
        (
            f"{page}: V8.9 Ring2 differs from "
            f"Notebook75 C1 geometry at "
            f"{ring2_xor} pixels."
        ),
    )

    require(
        output_diff == 0,
        (
            f"{page}: V8.9 does not reproduce "
            f"the Notebook75 C1 transformation "
            f"on the corrected V8.8 parent. "
            f"Different pixels: {output_diff}"
        ),
    )

    require(
        ring2_observed_mismatch == 0,
        (
            f"{page}: Ring2 does not contain "
            "exact observed bytes."
        ),
    )

    require(
        non_ring2_parent_mismatch == 0,
        (
            f"{page}: non-Ring2 regression "
            "from corrected V8.8 parent."
        ),
    )


    rows.append({
        "page":
            page,

        "ring2_pixels":
            int(expected_ring2.sum()),

        "ring2_geometry_xor":
            ring2_xor,

        "different_pixels_vs_expected_c1_transform":
            output_diff,

        "ring2_observed_mismatch":
            ring2_observed_mismatch,

        "non_ring2_parent_mismatch":
            non_ring2_parent_mismatch,
    })


c1_semantic = pd.DataFrame(rows)

c1_semantic.to_csv(
    RUN
    / "gates"
    / "notebook75_c1_semantic_reproduction.csv",
    index=False,
)


# ------------------------------------------------------------
# Pooled fail-closed assertion
# ------------------------------------------------------------

check_columns = [
    "ring2_geometry_xor",
    "different_pixels_vs_expected_c1_transform",
    "ring2_observed_mismatch",
    "non_ring2_parent_mismatch",
]

require(
    (
        c1_semantic[
            check_columns
        ].to_numpy() == 0
    ).all(),
    (
        "Notebook75 C1 semantic-reproduction "
        "gate failed."
    ),
)


print()
print("=" * 72)
print(
    "PASS: NOTEBOOK75 C1 SEMANTIC REPRODUCTION"
)
print("=" * 72)

print(
    "Pages:",
    len(c1_semantic),
)

print(
    "Ring2 pixels:",
    f"{int(c1_semantic.ring2_pixels.sum()):,}",
)

print(
    "Ring2 geometry XOR:",
    int(
        c1_semantic.ring2_geometry_xor.sum()
    ),
)

print(
    "Output differences from expected C1 transformation:",
    int(
        c1_semantic[
            "different_pixels_vs_expected_c1_transform"
        ].sum()
    ),
)

print(
    "Ring2 observed-byte mismatches:",
    int(
        c1_semantic.ring2_observed_mismatch.sum()
    ),
)

print(
    "Non-Ring2 corrected-V8.8 regressions:",
    int(
        c1_semantic.non_ring2_parent_mismatch.sum()
    ),
)


PASS: NOTEBOOK75 C1 SEMANTIC REPRODUCTION
Pages: 151
Ring2 pixels: 998,400
Ring2 geometry XOR: 0
Output differences from expected C1 transformation: 0
Ring2 observed-byte mismatches: 0
Non-Ring2 corrected-V8.8 regressions: 0


## Gate 3: exact deterministic replay

In [1]:
sample=roster.sort_values('page').head(min(12,len(roster))); replay=[]
for _,r in sample.iterrows():
 obs=cv2.imread(str(r.observed_path),0); seed=cv2.imread(str(r.seed_mask_path),0)
 a=op.run_page(obs,seed,page=str(r.page)); b=op.run_page(obs,seed,page=str(r.page))
 replay.append({'page':r.page,'corrected_exact':np.array_equal(a.corrected,b.corrected),'ring2_exact':np.array_equal(a.ring2_mask,b.ring2_mask),'p_print_exact':np.array_equal(a.p_print,b.p_print,equal_nan=True),'p_remove_exact':np.array_equal(a.p_remove,b.p_remove,equal_nan=True)})
replay=pd.DataFrame(replay); replay.to_csv(RUN/'replay'/'replay_results.csv',index=False)
require(replay.drop(columns='page').all().all(),'Replay gate failed.')
print('PASS: exact replay')


NameError: name 'roster' is not defined

## Gate 4: full 151-page HITL regeneration

In [ ]:
require(len(roster)==HITL_EXPECTED_PAGES,f'Expected 151 HITL pages, found {len(roster)}')
page_rows=[]
for _,r in roster.iterrows():
 page=str(r.page); obs=cv2.imread(str(r.observed_path),0); seed=cv2.imread(str(r.seed_path),0); z=op.run_page(obs,seed,page=page)
 d=RUN/'pages'/page; d.mkdir(parents=True,exist_ok=False)
 cv2.imwrite(str(d/'observed.png'),z.observed); cv2.imwrite(str(d/'authoritative_seed_mask.png'),seed); cv2.imwrite(str(d/'v8_8_parent.png'),z.parent_v88_corrected); cv2.imwrite(str(d/'v8_9.png'),z.corrected); cv2.imwrite(str(d/'ring2_mask.png'),z.ring2_mask.astype(np.uint8)*255); cv2.imwrite(str(d/'v8_9_delta.png'),z.v89_delta_mask.astype(np.uint8)*255)
 (d/'page_audit.json').write_text(json.dumps(z.audit,indent=2,default=str))
 page_rows.append({'page':page,'ring2_pixels':int(z.ring2_mask.sum()),'delta_pixels':int(z.v89_delta_mask.sum()),'non_ring2_parent_mismatch':int(np.sum(z.corrected[~z.ring2_mask]!=z.parent_v88_corrected[~z.ring2_mask])),'v89_sha256':sha256_file(d/'v8_9.png')})
pages=pd.DataFrame(page_rows); pages.to_csv(RUN/'audit'/'151_page_regeneration.csv',index=False)
require(len(pages)==151 and (pages.non_ring2_parent_mismatch==0).all(),'151-page regeneration gate failed.')
print('PASS: regenerated',len(pages),'HITL pages')


## Release verdict and upload package

In [ ]:
verdict={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'8.9','verdict':'PASS','gates':{'v8_8_parent_equivalence':True,'notebook75_c1_reproduction':True,'exact_replay':True,'hitl_151_page_regeneration':True},'pages':151,'operator_sha256':sha256_file(OPERATOR_SOURCE),'uses_clean_truth':False}
(RUN/'release_verdict.json').write_text(json.dumps(verdict,indent=2))
shutil.copy2(OPERATOR_SOURCE,RUN/'upload'/'v8_9_operator.py'); shutil.copy2(RUN/'release_verdict.json',RUN/'upload'/'release_verdict.json'); shutil.copy2(RUN/'audit'/'151_page_regeneration.csv',RUN/'upload'/'151_page_regeneration.csv'); shutil.copy2(RUN/'gates'/'v88_parent_equivalence.csv',RUN/'upload'/'v88_parent_equivalence.csv'); shutil.copy2(RUN/'gates'/'notebook75_c1_reproduction.csv',RUN/'upload'/'notebook75_c1_reproduction.csv'); shutil.make_archive(str(RUN/'notebook78_upload_bundle'),'zip',RUN/'upload')
print(json.dumps(verdict,indent=2)); print('COMPLETE:',RUN)
